# 14 - Baseline Forecasting Models

**Project:** AI Quantitative Trading & Strategy Optimisation System  
**Phase:** 7 — Baseline Forecasting Models  
**Status:** Research implementation; results below are generated from the local acquisitions when this notebook is executed.

## Research question and scope

Can simple statistical forecasts provide useful out-of-sample information for the existing Phase 6 price-return, direction and realised-volatility targets? This notebook establishes transparent reference forecasts before later ML/DL work. It does not define a trading strategy, BUY/SELL rule, position sizing, portfolio, backtest, execution process or profitability claim.

### Hypotheses

1. Zero and expanding historical location forecasts may be difficult to beat for forward returns.
2. Simple AR(1) forecasts of one-row log returns may add little beyond those benchmarks.
3. A small, pre-specified financial feature set may reduce return forecast error; exact-date peer/benchmark returns or past-only regime indicators may add incremental information.
4. Historical volatility may provide a useful reference for cumulative forward realised volatility.

These are research hypotheses, not expected outcomes.

## Data, targets and availability

The notebook reads existing local Yahoo Finance acquisitions for NG.L, SSE.L, SVT.L and FTSE 100. It reads `Close` and does not alter Raw files. The Phase 6 targets are retained unchanged: simple price return `Close[t+h] / Close[t] - 1`; direction at the zero neutral band; and cumulative, unannualised realised volatility `sqrt(sum(log_return[t+1:t+h] ** 2))`. Horizons are 5, 10 and 20 subsequent acquired rows. They are not independently verified exchange sessions, returns exclude dividends, and source adjustment/corporate-action semantics remain unresolved.

Close-based features are available only after that row's close. A forecast at origin `t` is for subsequent rows. Same-date cross-asset close returns are used only as after-close origin features; this does not imply same-session execution. Macroeconomic inputs are excluded because their historical release availability, vintages and licensing are not validated. FRED/ALFRED remains prohibited for training absent written authorisation.

## Validation and selection

The final 20% of each asset's acquired rows is reserved as a chronological holdout. Expanding walk-forward folds in the earlier 80% are used for development and selection. For horizon `h`, a training origin is retained only when its label endpoint is no later than the first validation origin; this purges intersecting future-return intervals. The expanding design never trains on observations after a validation block, so no post-validation training rows exist to embargo. An optional pre-validation embargo is implemented and tested, but the primary notebook uses zero additional embargo beyond horizon purging.

Every feature family uses identical fold origins. Scaling is estimated inside each training fold. No feature selection is done on the full sample. The feature family with the smallest pooled walk-forward MAE is selected separately by asset and horizon; only that family is evaluated once on the untouched final holdout. The holdout is not used to revise the family or settings.

## Methods and metrics

Univariate references are zero, training mean, training median and trailing 252-label mean. An AR(1) is fitted by least squares to one-row log returns and recursively forecasts the horizon return. Feature groups use ordinary least squares with training-only standardisation and no imputation: B (small financial set), BC (B plus exact-date peer/benchmark returns) and BCD (BC plus past-only regime states). This is a deliberately small statistical screen, not a comprehensive feature search. Volatility references are training median/mean and a trailing historical-volatility estimate. GARCH/ARIMA are not fitted unless residual/stationarity evidence and a declared dependency justify them; no added dependency is required here.

Return and volatility forecasts use MAE, RMSE and bias. Direction uses sign accuracy, precision, recall, F1 and a confusion matrix at the zero neutral band. Results are shown per asset, horizon and chronological fold. No minimum performance threshold has been approved; conclusions will be comparative and descriptive.

In [9]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs" / "assets").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecasting.benchmarks import (
    autoregressive_horizon_return_forecast,
    fit_ar1_parameters,
    constant_return_forecast,
    historical_volatility_forecast,
    moving_average_return_forecast,
)
from src.forecasting.garch import (
    fit_garch11,
    forecast_garch11_cumulative_volatility,
    update_garch11_state,
)
from src.forecasting.linear import fit_ols_forecast
from src.forecasting.metrics import directional_metrics, regression_metrics
from src.forecasting.validation import chronological_holdout, expanding_window_splits
from src.features.cross_asset import build_cross_asset_return_panel
from src.features.financial import build_financial_features, log_returns
from src.regimes import build_regime_indicators
from src.targets import DEFAULT_HORIZONS, build_forward_targets
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.stattools import adfuller, kpss

HORIZONS = DEFAULT_HORIZONS
HOLDOUT_FRACTION = 0.20
NEUTRAL_THRESHOLD = 0.0
print(f"Horizons: {HORIZONS}; final holdout fraction: {HOLDOUT_FRACTION:.0%}")

ImportError: cannot import name 'fit_ar1_parameters' from 'src.forecasting.benchmarks' (/home/anderjcruz/projects/Quantitative-Trading-Strategy/src/forecasting/benchmarks.py)

## 1. Load and inventory the acquired market series

Each configured symbol must resolve to one local Parquet acquisition. The loader reads files without writing to the data directories. Missing values and differing calendars are retained; no forward-fill or interpolation is applied.

In [ ]:
asset_configs = sorted((PROJECT_ROOT / "configs" / "assets").glob("*.yaml"))
market_data = {}
source_inventory = []
for config_path in asset_configs:
    with config_path.open(encoding="utf-8") as stream:
        config = yaml.safe_load(stream)
    symbol = config["provider"]["symbol"]
    candidates = sorted(
        (PROJECT_ROOT / "data" / "raw" / "market" / "yahoo_finance" / symbol)
        .glob(f"acquisition_*/{symbol}.parquet")
    )
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one local Parquet acquisition for {symbol}; found {len(candidates)}")
    frame = pd.read_parquet(candidates[0])
    if "Date" in frame.columns:
        frame = frame.set_index("Date")
    frame.index = pd.to_datetime(frame.index)
    frame.index.name = "forecast_origin_observation_date"
    if frame.index.has_duplicates or not frame.index.is_monotonic_increasing:
        raise ValueError(f"Dates for {symbol} must be unique and chronological")
    if "Close" not in frame:
        raise ValueError(f"Close is absent for {symbol}")
    market_data[symbol] = frame
    source_inventory.append({
        "symbol": symbol,
        "source_file": candidates[0].relative_to(PROJECT_ROOT).as_posix(),
        "price_unit": config["asset"].get("price_unit", "not configured"),
        "unit_validation": config["asset"].get("price_unit_validation_status", "not recorded"),
        "rows": len(frame),
        "first_date": frame.index.min(),
        "last_date": frame.index.max(),
        "missing_close": int(frame["Close"].isna().sum()),
    })
source_inventory = pd.DataFrame(source_inventory).sort_values("symbol").reset_index(drop=True)
source_inventory

,symbol,source_file,price_unit,unit_validation,rows,first_date,last_date,missing_close
0,NG.L,data/raw/market/yahoo_finance/NG.L/acquisition...,GBp,pending,7890,1995-12-11,2026-11-02,1
1,SSE.L,data/raw/market/yahoo_finance/SSE.L/acquisitio...,GBp,pending,9060,1991-06-17,2026-11-02,1
2,SVT.L,data/raw/market/yahoo_finance/SVT.L/acquisitio...,GBp,pending,9455,1989-12-11,2026-11-02,1
3,^FTSE,data/raw/market/yahoo_finance/^FTSE/acquisitio...,index_points,not_applicable,11004,1984-01-03,2026-11-02,184


## 2. Build existing targets and pre-specified feature groups

Group B uses five interpretable Phase 6 features. Group C adds the same-date one-row returns of the other configured assets, aligned by exact dates. Group D adds one-hot past-only trend and relative-volatility state indicators. These groups are pre-specified and kept separate from target columns. They are not selected using full-sample correlations.

In [ ]:
prices = {symbol: frame["Close"].astype(float) for symbol, frame in market_data.items()}
cross_asset = build_cross_asset_return_panel(prices, lookbacks=(1,))

feature_data = {}
target_data = {}
for symbol, frame in market_data.items():
    financial = build_financial_features(frame, lookbacks=(5, 20))[[
        "simple_return_1", "momentum_5", "momentum_20",
        "realised_volatility_20", "drawdown_20",
    ]].copy()
    peer_columns = [column for column in cross_asset.columns if column.endswith("__return_1") and not column.startswith(f"{symbol}__")]
    group_b = financial.rename(columns={
        "simple_return_1": "asset_return_1",
        "momentum_5": "asset_momentum_5",
        "momentum_20": "asset_momentum_20",
        "realised_volatility_20": "asset_realised_volatility_20",
        "drawdown_20": "asset_drawdown_20",
    })
    group_bc = group_b.join(cross_asset[peer_columns], how="left")
    regimes = build_regime_indicators(frame, volatility_window=20, threshold_window=252, trend_windows=(50, 200))
    regime_categories = pd.DataFrame(index=frame.index)
    for column, allowed in (("volatility_regime", ["low_volatility", "typical_volatility", "high_volatility"]),
                            ("trend_regime", ["downtrend", "mixed_trend", "uptrend"])):
        categorical = pd.Categorical(regimes[column], categories=allowed)
        encoded = pd.get_dummies(categorical, prefix=column, dtype=float)
        encoded.index = frame.index
        regime_categories = regime_categories.join(encoded)
    group_bcd = group_bc.join(regime_categories, how="left")
    feature_data[symbol] = {"B_financial": group_b, "BC_cross_asset": group_bc, "BCD_regime": group_bcd}
    target_data[symbol] = build_forward_targets(frame, horizons=HORIZONS, neutral_thresholds=(NEUTRAL_THRESHOLD,))

print("Candidate groups:", {symbol: {name: len(values.columns) for name, values in groups.items()} for symbol, groups in feature_data.items()})

Candidate groups: {'^FTSE': {'B_financial': 5, 'BC_cross_asset': 8, 'BCD_regime': 14}, 'NG.L': {'B_financial': 5, 'BC_cross_asset': 8, 'BCD_regime': 14}, 'SSE.L': {'B_financial': 5, 'BC_cross_asset': 8, 'BCD_regime': 14}, 'SVT.L': {'B_financial': 5, 'BC_cross_asset': 8, 'BCD_regime': 14}}


## 3. Statistical diagnostics for AR and volatility candidates

ADF and KPSS test the one-row log returns using only an initial training prefix per asset. Ljung–Box tests AR(1) residual autocorrelation; ARCH-LM tests conditional heteroskedasticity in those residuals. The tests are descriptive and are not repeated on the final holdout. Missing return rows are omitted for these tests; the GARCH estimator itself retains acquired-row gaps by propagating expected variance.

In [ ]:
def _ar1_residuals(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)
    parameters = fit_ar1_parameters(pd.Series(values))
    residuals = values[1:] - (parameters[0] + parameters[1] * values[:-1])
    return parameters, residuals


diagnostic_rows = []
for symbol, frame in market_data.items():
    log_return_series = log_returns(frame["Close"])
    development_end = int(np.floor(len(frame) * (1 - HOLDOUT_FRACTION)))
    initial_train = max(int(development_end * 0.45), max(HORIZONS) + 30)
    fit_end = initial_train - max(HORIZONS)
    training_returns = log_return_series.iloc[:fit_end + 1]
    finite_returns = training_returns.dropna()
    ar_parameters, ar_residuals = _ar1_residuals(finite_returns)
    adf_result = adfuller(finite_returns, maxlag=10, autolag="AIC")
    kpss_result = kpss(finite_returns, regression="c", nlags="auto")
    ljung_box_p = acorr_ljungbox(ar_residuals, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    arch_lm_p = het_arch(ar_residuals, nlags=10)[1]
    diagnostic_rows.append({
        "symbol": symbol,
        "training_rows": len(training_returns),
        "finite_returns": len(finite_returns),
        "ar1_phi": ar_parameters[1],
        "adf_pvalue_unit_root_null": adf_result[1],
        "kpss_pvalue_stationarity_null": kpss_result[1],
        "ljung_box_10_pvalue_residual_autocorrelation_null": ljung_box_p,
        "arch_lm_10_pvalue_no_arch_null": arch_lm_p,
    })

ar_diagnostics = pd.DataFrame(diagnostic_rows).set_index("symbol")
ar_diagnostics.round(5)

/tmp/ipykernel_26948/3281911849.py:17: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_result = adfuller(finite_returns, maxlag=10, autolag="AIC")
/tmp/ipykernel_26948/3281911849.py:18: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_result = kpss(finite_returns, regression="c", nlags="auto")
/tmp/ipykernel_26948/3281911849.py:18: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default be

,training_rows,finite_returns,ar1_phi,adf_pvalue_unit_root_null,kpss_pvalue_stationarity_null,ljung_box_10_pvalue_residual_autocorrelation_null,arch_lm_10_pvalue_no_arch_null
symbol,,,,,,,
^FTSE,3942,3725,0.07946,0.0,0.1,0.01592,0.0
NG.L,2821,2820,-0.29109,0.0,0.1,0.00000,0.0
SSE.L,3242,3241,-0.00512,0.0,0.1,0.58035,0.0
SVT.L,3384,3383,0.03567,0.0,0.1,0.18551,0.0


## 4. Walk-forward development evaluation

For each asset and horizon, the first 80% of rows forms the development region. Expanding validation blocks cover its later portion. Models are refit for each fold using only eligible training rows. Rows with unavailable targets or incomplete feature vectors are omitted within the relevant fold; nothing is imputed. A fold with insufficient complete history is reported by the result count rather than silently backfilled.

In [ ]:
def _fold_predictions(symbol, horizon):
    frame = market_data[symbol]
    n_rows = len(frame)
    development_end = int(np.floor(n_rows * (1 - HOLDOUT_FRACTION)))
    initial = max(int(development_end * 0.45), horizon + 30)
    validation_size = max(int(development_end * 0.10), 20)
    folds = expanding_window_splits(
        development_end,
        initial_train_size=initial,
        validation_size=validation_size,
        horizon=horizon,
        step_size=validation_size,
    )
    target = target_data[symbol][f"forward_return_{horizon}"]
    vol_target = target_data[symbol][f"forward_realised_volatility_{horizon}"]
    one_row_log_returns = log_returns(frame["Close"])
    records = []
    for fold_number, (train_pos, valid_pos) in enumerate(folds, start=1):
        train_idx = frame.index[train_pos]
        valid_idx = frame.index[valid_pos]
        common_train_mask = target.iloc[train_pos].notna().copy()
        common_valid_mask = target.iloc[valid_pos].notna().copy()
        for candidate_features in feature_data[symbol].values():
            common_train_mask &= candidate_features.iloc[train_pos].notna().all(axis=1)
            common_valid_mask &= candidate_features.iloc[valid_pos].notna().all(axis=1)
        y_train = target.iloc[train_pos].loc[common_train_mask]
        if len(y_train) < 30:
            continue
        y_valid = target.iloc[valid_pos]
        valid_dates = valid_idx[common_valid_mask.to_numpy()]
        y_valid = y_valid.loc[valid_dates]
        if y_valid.empty:
            continue
        baseline_values = {
            "zero": constant_return_forecast(y_train, statistic="zero"),
            "historical_mean": constant_return_forecast(y_train, statistic="mean"),
            "historical_median": constant_return_forecast(y_train, statistic="median"),
            "moving_average_252": moving_average_return_forecast(y_train, window=min(252, len(y_train))),
        }
        ar_history = one_row_log_returns.loc[:train_idx.max()]
        ar_parameters = fit_ar1_parameters(ar_history) if ar_history.notna().sum() >= 30 else None
        for model_name, prediction in baseline_values.items():
            for date, actual in y_valid.items():
                records.append({"symbol": symbol, "horizon": horizon, "fold": fold_number,
                                "model": model_name, "date": date, "actual": actual,
                                "forecast": prediction})
        if ar_parameters is not None:
            for date, actual in y_valid.items():
                prediction = autoregressive_horizon_return_forecast(
                    one_row_log_returns.loc[:date], horizon=horizon, parameters=ar_parameters
                )
                records.append({"symbol": symbol, "horizon": horizon, "fold": fold_number,
                                "model": "AR1_log_returns", "date": date, "actual": actual,
                                "forecast": prediction})

        for group_name, group_features in feature_data[symbol].items():
            x_train = group_features.iloc[train_pos]
            train_mask = common_train_mask
            x_valid = group_features.loc[valid_dates]
            if train_mask.sum() < max(30, group_features.shape[1] + 5) or x_valid.empty:
                continue
            predictions = fit_ols_forecast(
                x_train.loc[train_mask], target.iloc[train_pos].loc[train_mask],
                x_valid,
            )
            for date, prediction in predictions.items():
                records.append({"symbol": symbol, "horizon": horizon, "fold": fold_number,
                                "model": group_name, "date": date,
                                "actual": target.loc[date], "forecast": prediction})

        # Volatility benchmarks share the same chronological fold origins.
        vol_train = vol_target.iloc[train_pos].dropna()
        vol_valid = vol_target.iloc[valid_pos].dropna()
        if len(vol_train) >= 30 and not vol_valid.empty:
            garch_train_history = one_row_log_returns.iloc[:train_pos[-1] + 1]
            try:
                garch_parameters = fit_garch11(garch_train_history)
            except (ValueError, RuntimeError) as error:
                garch_parameters = None
                garch_fit_error = str(error)
            garch_origin_position = int(train_pos[-1])
            for date, actual in vol_valid.items():
                history = one_row_log_returns.loc[:date]
                volatility_window = min(252, len(history))
                if volatility_window < 2 or history.tail(volatility_window).isna().any():
                    continue
                volatility_forecasts = {
                    "volatility_historical_mean": float(vol_train.mean()),
                    "volatility_historical_median": float(vol_train.median()),
                    "volatility_rolling_historical": historical_volatility_forecast(
                        history, window=volatility_window, horizon=horizon
                    ),
                }
                if garch_parameters is not None:
                    current_position = frame.index.get_loc(date)
                    new_returns = one_row_log_returns.iloc[garch_origin_position + 1:current_position + 1]
                    current_garch_state = update_garch11_state(garch_parameters, new_returns)
                    volatility_forecasts["volatility_GARCH11"] = forecast_garch11_cumulative_volatility(
                        current_garch_state, horizon=horizon
                    )
                for model_name, prediction in volatility_forecasts.items():
                    records.append({"symbol": symbol, "horizon": horizon, "fold": fold_number,
                                    "model": model_name, "target_type": "volatility", "date": date,
                                    "actual": actual, "forecast": prediction})
    return pd.DataFrame(records)

fold_results = pd.concat(
    [_fold_predictions(symbol, horizon) for symbol in market_data for horizon in HORIZONS],
    ignore_index=True,
)
fold_results["target_type"] = fold_results.get("target_type", "return")
fold_results["target_type"] = fold_results["target_type"].fillna("return")
print(f"Walk-forward forecast records: {len(fold_results):,}")
fold_results.head()

Walk-forward forecast records: 539,148


,symbol,horizon,fold,model,date,actual,forecast,target_type
0,^FTSE,5,1,zero,1999-03-10,-0.016166,0.0,return
1,^FTSE,5,1,zero,1999-03-11,-0.034945,0.0,return
2,^FTSE,5,1,zero,1999-03-12,-0.018942,0.0,return
3,^FTSE,5,1,zero,1999-03-15,-0.008700,0.0,return
4,^FTSE,5,1,zero,1999-03-16,-0.022799,0.0,return


## 5. Development metrics and candidate selection

Return/volatility results use regression errors. Directional scores threshold forecasts and actual returns at zero; neutral outcomes therefore count as non-positive in this binary confusion matrix, while sign accuracy retains zero as its own sign. Feature families are ranked by pooled development MAE within each asset and horizon. This ranking is exploratory and subject to selection optimism; the final chronological holdout is reserved to assess the selected family.

In [ ]:
def _metric_row(group, model_name):
    scores = regression_metrics(group["actual"], group["forecast"])
    if not str(model_name).startswith("volatility_"):
        directional = directional_metrics(group["actual"], group["forecast"])
        scores.update({key: directional[key] for key in (
            "directional_accuracy", "precision_positive", "recall_positive", "f1_positive"
        )})
    return scores


def _summarise(frame, keys):
    rows = []
    for key, group in frame.groupby(keys, sort=True):
        key = key if isinstance(key, tuple) else (key,)
        row = dict(zip(keys, key))
        row.update(_metric_row(group, row["model"]))
        rows.append(row)
    return pd.DataFrame(rows)

cv_summary = _summarise(
    fold_results, ["symbol", "horizon", "target_type", "model"]
)
cv_summary.sort_values(["target_type", "symbol", "horizon", "mae"]).head(30)

,symbol,horizon,target_type,model,mae,mse,rmse,forecast_bias,n_observations,directional_accuracy,precision_positive,recall_positive,f1_positive
4,NG.L,5,return,historical_mean,0.020190,0.000779,0.027905,0.000905,3470,0.536599,0.536599,1.000000,0.698425
5,NG.L,5,return,historical_median,0.020193,0.000779,0.027916,0.001199,3470,0.536599,0.536599,1.000000,0.698425
0,NG.L,5,return,AR1_log_returns,0.020208,0.000772,0.027786,0.000419,3470,0.520749,0.540958,0.734157,0.622921
2,NG.L,5,return,BC_cross_asset,0.020212,0.000768,0.027713,0.001310,3470,0.531988,0.552101,0.705693,0.619519
3,NG.L,5,return,B_financial,0.020230,0.000768,0.027719,0.001307,3470,0.527666,0.550737,0.682062,0.609405
7,NG.L,5,return,zero,0.020274,0.000778,0.027894,-0.000841,3470,0.007781,0.000000,0.000000,0.000000
1,NG.L,5,return,BCD_regime,0.020376,0.000777,0.027872,0.001724,3470,0.519597,0.542642,0.697100,0.610249
6,NG.L,5,return,moving_average_252,0.020405,0.000791,0.028124,0.000381,3470,0.506340,0.530111,0.718582,0.610123
12,NG.L,10,return,AR1_log_returns,0.027670,0.001363,0.036915,0.000944,3470,0.537176,0.543152,0.874464,0.670092
17,NG.L,10,return,historical_median,0.027730,0.001373,0.037050,0.001944,3470,0.537176,0.537176,1.000000,0.698913


### Performance stability across chronological folds

The following view reports fold-level MAE for the univariate references and feature groups. It exposes whether pooled results depend on one validation period.

In [ ]:
fold_stability = _summarise(
    fold_results, ["symbol", "horizon", "target_type", "fold", "model"]
)
fold_stability.head(40)

,symbol,horizon,target_type,fold,model,mae,mse,rmse,forecast_bias,n_observations,directional_accuracy,precision_positive,recall_positive,f1_positive
0,NG.L,5,return,1,AR1_log_returns,0.025824,0.001263,0.035538,0.002322,631,0.521395,0.529412,0.717791,0.609375
1,NG.L,5,return,1,BCD_regime,0.025798,0.001237,0.035173,0.005205,631,0.507132,0.517021,0.745399,0.610553
2,NG.L,5,return,1,BC_cross_asset,0.025607,0.001232,0.035094,0.003709,631,0.543582,0.544843,0.745399,0.629534
3,NG.L,5,return,1,B_financial,0.025606,0.001229,0.035064,0.003821,631,0.540412,0.545244,0.720859,0.620872
4,NG.L,5,return,1,historical_mean,0.025965,0.001290,0.035912,0.002760,631,0.516640,0.516640,1.000000,0.681296
5,NG.L,5,return,1,historical_median,0.025964,0.001288,0.035882,0.002331,631,0.516640,0.516640,1.000000,0.681296
6,NG.L,5,return,1,moving_average_252,0.026123,0.001316,0.036278,0.005836,631,0.516640,0.516640,1.000000,0.681296
7,NG.L,5,return,1,zero,0.025997,0.001282,0.035812,0.000650,631,0.009509,0.000000,0.000000,0.000000
8,NG.L,5,return,2,AR1_log_returns,0.019901,0.000673,0.025951,-0.000848,629,0.511924,0.547393,0.665706,0.600780
9,NG.L,5,return,2,BCD_regime,0.020226,0.000693,0.026325,-0.000611,629,0.524642,0.563492,0.613833,0.587586


## 6. One-time final holdout assessment

The selected feature group is refit on development rows whose targets end no later than the first holdout origin. Baselines are computed from the same eligible development labels. The holdout predictions are generated once for the chosen group per asset and horizon. No holdout result feeds back into feature or model selection.

In [ ]:
def _holdout_evaluation(symbol, horizon):
    frame = market_data[symbol]
    development_pos, holdout_pos = chronological_holdout(
        len(frame), holdout_fraction=HOLDOUT_FRACTION, horizon=horizon
    )
    target = target_data[symbol][f"forward_return_{horizon}"]
    common_train_mask = target.iloc[development_pos].notna().copy()
    common_test_mask = target.iloc[holdout_pos].notna().copy()
    for candidate_features in feature_data[symbol].values():
        common_train_mask &= candidate_features.iloc[development_pos].notna().all(axis=1)
        common_test_mask &= candidate_features.iloc[holdout_pos].notna().all(axis=1)
    common_train_positions = development_pos[common_train_mask.to_numpy()]
    target_train = target.iloc[common_train_positions].dropna()
    holdout_idx = frame.index[holdout_pos]
    target_test = target.loc[holdout_idx[common_test_mask.to_numpy()]].dropna()
    if len(target_train) < 30 or target_test.empty:
        return []
    candidates = cv_summary.query(
        "symbol == @symbol and horizon == @horizon and target_type == 'return' and model in ['B_financial', 'BC_cross_asset', 'BCD_regime']"
    ).sort_values("mae")
    selected_group = candidates.iloc[0]["model"] if not candidates.empty else None
    baseline_predictions = {
        "zero": constant_return_forecast(target_train, statistic="zero"),
        "historical_mean": constant_return_forecast(target_train, statistic="mean"),
        "historical_median": constant_return_forecast(target_train, statistic="median"),
        "moving_average_252": moving_average_return_forecast(target_train, window=min(252, len(target_train))),
    }
    all_log_returns = log_returns(frame["Close"])
    ar_history = all_log_returns.iloc[:development_pos[-1] + 1]
    ar_parameters = fit_ar1_parameters(ar_history) if ar_history.notna().sum() >= 30 else None
    records = []
    for model_name, value in baseline_predictions.items():
        for date, actual in target_test.items():
            records.append({"symbol": symbol, "horizon": horizon, "model": model_name,
                            "date": date, "actual": actual, "forecast": value})
    if ar_parameters is not None:
        for date, actual in target_test.items():
            prediction = autoregressive_horizon_return_forecast(
                all_log_returns.loc[:date], horizon=horizon, parameters=ar_parameters
            )
            records.append({"symbol": symbol, "horizon": horizon, "model": "AR1_log_returns",
                            "date": date, "actual": actual, "forecast": prediction})
    if selected_group:
        features = feature_data[symbol][selected_group]
        train_x = features.iloc[common_train_positions]
        train_target = target.iloc[common_train_positions]
        test_x = features.loc[target_test.index]
        if len(train_target) >= max(30, features.shape[1] + 5) and not test_x.empty:
            predictions = fit_ols_forecast(
                train_x, train_target,
                test_x,
            )
            for date, forecast in predictions.items():
                records.append({"symbol": symbol, "horizon": horizon,
                                "model": f"selected_{selected_group}", "date": date,
                                "actual": target.loc[date], "forecast": forecast})
    return records


holdout_records = [
    row for symbol in market_data for horizon in HORIZONS
    for row in _holdout_evaluation(symbol, horizon)
]
holdout_results = pd.DataFrame(holdout_records)
holdout_results["target_type"] = "return"
holdout_summary = _summarise(
    holdout_results, ["symbol", "horizon", "model"]
)
holdout_summary.sort_values(["symbol", "horizon", "mae"])

,symbol,horizon,model,mae,mse,rmse,forecast_bias,n_observations,directional_accuracy,precision_positive,recall_positive,f1_positive
3,NG.L,5,moving_average_252,0.021528,0.000843,0.029038,0.000940,1570,0.549682,0.549682,1.000000,0.709412
2,NG.L,5,historical_median,0.021534,0.000843,0.029033,0.000759,1570,0.549682,0.549682,1.000000,0.709412
1,NG.L,5,historical_mean,0.021566,0.000842,0.029023,0.000020,1570,0.549682,0.549682,1.000000,0.709412
0,NG.L,5,AR1_log_returns,0.021633,0.000848,0.029123,-0.000434,1570,0.536943,0.563367,0.705678,0.626543
4,NG.L,5,selected_BC_cross_asset,0.021657,0.000863,0.029379,-0.000017,1570,0.531210,0.566701,0.634994,0.598907
...,...,...,...,...,...,...,...,...,...,...,...,...
68,^FTSE,20,historical_median,0.027041,0.001620,0.040255,0.003709,2139,0.578775,0.578775,1.000000,0.733195
67,^FTSE,20,historical_mean,0.027055,0.001607,0.040084,0.000214,2139,0.578775,0.578775,1.000000,0.733195
66,^FTSE,20,AR1_log_returns,0.027064,0.001607,0.040084,0.000063,2139,0.578775,0.578775,1.000000,0.733195
69,^FTSE,20,moving_average_252,0.027177,0.001609,0.040112,-0.001512,2139,0.578775,0.578775,1.000000,0.733195


### Compact development and holdout comparison

This table compares the best naive return benchmark and the feature family selected on development folds, then reports both on the same untouched holdout dates. The relative MAE change is descriptive; a positive value means the selected feature family had higher holdout MAE.

In [ ]:
def _return_decision_row(symbol, horizon):
    development = cv_summary.query(
        "symbol == @symbol and horizon == @horizon and target_type == 'return'"
    )
    holdout = holdout_summary.query("symbol == @symbol and horizon == @horizon")
    naive_names = ["zero", "historical_mean", "historical_median", "moving_average_252", "AR1_log_returns"]
    best_naive_dev = development[development.model.isin(naive_names)].sort_values("mae").iloc[0]
    best_feature_dev = development[development.model.isin(
        ["B_financial", "BC_cross_asset", "BCD_regime"]
    )].sort_values("mae").iloc[0]
    selected_holdout = holdout[holdout.model == f"selected_{best_feature_dev.model}"]
    naive_holdout = holdout[holdout.model.isin(naive_names)].sort_values("mae")
    if selected_holdout.empty or naive_holdout.empty:
        return None
    selected_holdout = selected_holdout.iloc[0]
    best_naive_holdout = naive_holdout.iloc[0]
    best_direction_naive = naive_holdout.sort_values("directional_accuracy", ascending=False).iloc[0]
    return {
        "symbol": symbol, "horizon": horizon,
        "best_development_naive": best_naive_dev.model,
        "best_development_naive_mae": best_naive_dev.mae,
        "selected_feature_group": best_feature_dev.model,
        "feature_development_mae": best_feature_dev.mae,
        "best_holdout_naive": best_naive_holdout.model,
        "best_holdout_naive_mae": best_naive_holdout.mae,
        "selected_feature_holdout_mae": selected_holdout.mae,
        "feature_holdout_mae_change_vs_best_naive": (
            selected_holdout.mae / best_naive_holdout.mae - 1.0
        ),
        "best_direction_naive": best_direction_naive.model,
        "best_direction_naive_accuracy": best_direction_naive.directional_accuracy,
        "selected_feature_directional_accuracy": selected_holdout.directional_accuracy,
        "directional_accuracy_change": (
            selected_holdout.directional_accuracy - best_direction_naive.directional_accuracy
        ),
        "holdout_rows_naive": best_naive_holdout.n_observations,
        "holdout_rows_feature": selected_holdout.n_observations,
    }

return_decision_summary = pd.DataFrame([
    row for symbol in market_data for horizon in HORIZONS
    if (row := _return_decision_row(symbol, horizon)) is not None
])
return_decision_summary.round(6)

,symbol,horizon,best_development_naive,best_development_naive_mae,selected_feature_group,feature_development_mae,best_holdout_naive,best_holdout_naive_mae,selected_feature_holdout_mae,feature_holdout_mae_change_vs_best_naive,best_direction_naive,best_direction_naive_accuracy,selected_feature_directional_accuracy,directional_accuracy_change,holdout_rows_naive,holdout_rows_feature
0,^FTSE,5,historical_mean,0.017541,B_financial,0.017719,historical_median,0.014208,0.014168,-0.002810,historical_median,0.562932,0.538036,-0.024896,2169,2169
1,^FTSE,10,AR1_log_returns,0.023908,B_financial,0.024277,historical_median,0.019989,0.019660,-0.016463,historical_median,0.570635,0.556739,-0.013895,2159,2159
2,^FTSE,20,AR1_log_returns,0.032084,B_financial,0.033717,historical_median,0.027041,0.026402,-0.023613,historical_median,0.578775,0.553530,-0.025245,2139,2139
3,NG.L,5,historical_mean,0.020190,BC_cross_asset,0.020212,moving_average_252,0.021528,0.021657,0.005967,moving_average_252,0.549682,0.531210,-0.018471,1570,1570
4,NG.L,10,AR1_log_returns,0.027670,B_financial,0.027763,moving_average_252,0.029546,0.029895,0.011809,moving_average_252,0.585304,0.538019,-0.047284,1565,1565
5,NG.L,20,AR1_log_returns,0.037947,B_financial,0.037832,moving_average_252,0.043632,0.043663,0.000709,moving_average_252,0.546624,0.529260,-0.017363,1555,1555
6,SSE.L,5,historical_mean,0.019391,B_financial,0.019528,historical_median,0.026870,0.027286,0.015492,AR1_log_returns,0.547118,0.493348,-0.053769,1804,1804
7,SSE.L,10,historical_mean,0.026605,B_financial,0.027087,historical_median,0.036160,0.037463,0.036030,AR1_log_returns,0.566982,0.469705,-0.097276,1799,1799
8,SSE.L,20,AR1_log_returns,0.037336,B_financial,0.037486,historical_median,0.053599,0.055434,0.034230,AR1_log_returns,0.580771,0.475685,-0.105087,1789,1789
9,SVT.L,5,AR1_log_returns,0.021499,B_financial,0.021527,historical_median,0.022925,0.022924,-0.000031,AR1_log_returns,0.533192,0.519384,-0.013808,1883,1883


In [ ]:
def _holdout_volatility(symbol, horizon):
    frame = market_data[symbol]
    development_pos, holdout_pos = chronological_holdout(
        len(frame), holdout_fraction=HOLDOUT_FRACTION, horizon=horizon
    )
    target = target_data[symbol][f"forward_realised_volatility_{horizon}"]
    train_values = target.iloc[development_pos].dropna()
    test_values = target.iloc[holdout_pos].dropna()
    if len(train_values) < 30 or test_values.empty:
        return []
    returns = log_returns(frame["Close"])
    garch_history = returns.iloc[:development_pos[-1] + 1]
    try:
        garch_parameters = fit_garch11(garch_history)
    except (ValueError, RuntimeError):
        garch_parameters = None
    garch_origin_position = int(development_pos[-1])
    rows = []
    for date, actual in test_values.items():
        history = returns.loc[:date]
        window = min(252, len(history))
        if window < 2 or history.tail(window).isna().any():
            continue
        predictions = {
            "volatility_historical_mean": float(train_values.mean()),
            "volatility_historical_median": float(train_values.median()),
            "volatility_rolling_historical": historical_volatility_forecast(
                history, window=window, horizon=horizon
            ),
        }
        if garch_parameters is not None:
            current_position = frame.index.get_loc(date)
            new_returns = returns.iloc[garch_origin_position + 1:current_position + 1]
            current_garch_state = update_garch11_state(garch_parameters, new_returns)
            predictions["volatility_GARCH11"] = forecast_garch11_cumulative_volatility(
                current_garch_state, horizon=horizon
            )
        for model_name, forecast in predictions.items():
            rows.append({"symbol": symbol, "horizon": horizon, "model": model_name,
                         "date": date, "actual": actual, "forecast": forecast})
    return rows

volatility_holdout_results = pd.DataFrame([
    row for symbol in market_data for horizon in HORIZONS
    for row in _holdout_volatility(symbol, horizon)
])
volatility_holdout_summary = _summarise(
    volatility_holdout_results, ["symbol", "horizon", "model"]
)
volatility_holdout_summary.sort_values(["symbol", "horizon", "mae"])

,symbol,horizon,model,mae,mse,rmse,forecast_bias,n_observations
2,NG.L,5,volatility_historical_median,0.008895,0.000185,0.013588,-0.001907,1572
0,NG.L,5,volatility_GARCH11,0.010873,0.000207,0.014374,0.005674,1572
1,NG.L,5,volatility_historical_mean,0.010946,0.000208,0.014409,0.005162,1572
3,NG.L,5,volatility_rolling_historical,0.011295,0.000232,0.015241,0.004788,1572
6,NG.L,10,volatility_historical_median,0.010366,0.000265,0.016278,-0.001582,1567
4,NG.L,10,volatility_GARCH11,0.013050,0.000320,0.017896,0.007487,1567
7,NG.L,10,volatility_rolling_historical,0.013540,0.000355,0.018844,0.005441,1567
5,NG.L,10,volatility_historical_mean,0.014105,0.000331,0.018187,0.008264,1567
10,NG.L,20,volatility_historical_median,0.012803,0.000394,0.019845,-0.000969,1557
11,NG.L,20,volatility_rolling_historical,0.017540,0.000581,0.024107,0.006507,1557


### Holdout confusion matrices for directional forecasts

The confusion matrix treats positive returns as the positive class and zero/negative returns as the non-positive class, matching the notebook's documented binary precision/recall convention. The sign accuracy remains separately reported.

In [ ]:
direction_rows = []
for row in return_decision_summary.to_dict("records"):
    symbol, horizon = row["symbol"], row["horizon"]
    holdout_group = holdout_results.query("symbol == @symbol and horizon == @horizon")
    chosen_feature = holdout_group[holdout_group.model == f"selected_{row['selected_feature_group']}"]
    naive_group = holdout_group[holdout_group.model.isin(
        ["zero", "historical_mean", "historical_median", "moving_average_252", "AR1_log_returns"]
    )]
    best_naive_model = naive_group.groupby("model").apply(
        lambda group: directional_metrics(group["actual"], group["forecast"])["directional_accuracy"]
    ).sort_values(ascending=False).index[0]
    best_naive = naive_group[naive_group.model == best_naive_model]
    for model_name, group in ((f"selected_{row['selected_feature_group']}", chosen_feature),
                              (best_naive_model, best_naive)):
        scores = directional_metrics(group["actual"], group["forecast"])
        direction_rows.append({
            "symbol": symbol, "horizon": horizon, "model": model_name,
            "directional_accuracy": scores["directional_accuracy"],
            "precision_positive": scores["precision_positive"],
            "recall_positive": scores["recall_positive"],
            "f1_positive": scores["f1_positive"],
            "confusion_matrix_negative_positive": scores["confusion_matrix_negative_positive"],
        })
holdout_direction_summary = pd.DataFrame(direction_rows)
holdout_direction_summary

,symbol,horizon,model,directional_accuracy,precision_positive,recall_positive,f1_positive,confusion_matrix_negative_positive
0,^FTSE,5,selected_B_financial,0.538036,0.594153,0.565930,0.579698,"[[476, 472], [530, 691]]"
1,^FTSE,5,AR1_log_returns,0.562932,0.562932,1.000000,0.720354,"[[0, 948], [0, 1221]]"
2,^FTSE,10,selected_B_financial,0.556739,0.623318,0.564123,0.592245,"[[507, 420], [537, 695]]"
3,^FTSE,10,AR1_log_returns,0.570635,0.570635,1.000000,0.726629,"[[0, 927], [0, 1232]]"
4,^FTSE,20,selected_B_financial,0.553530,0.600855,0.680937,0.638395,"[[341, 560], [395, 843]]"
5,^FTSE,20,AR1_log_returns,0.578775,0.578775,1.000000,0.733195,"[[0, 901], [0, 1238]]"
6,NG.L,5,selected_BC_cross_asset,0.531210,0.566701,0.634994,0.598907,"[[288, 419], [315, 548]]"
7,NG.L,5,historical_mean,0.549682,0.549682,1.000000,0.709412,"[[0, 707], [0, 863]]"
8,NG.L,10,selected_B_financial,0.538019,0.593269,0.673581,0.630879,"[[226, 423], [299, 617]]"
9,NG.L,10,historical_mean,0.585304,0.585304,1.000000,0.738412,"[[0, 649], [0, 916]]"


## 7. Volatility forecast results

Volatility target values are the Phase 6 cumulative realised-volatility definition and are not annualised. The rolling historical forecast assumes recent variance persists over the next horizon; it is a simple benchmark, not a fitted conditional-volatility model. Compare it with training mean and median using identical walk-forward periods.

In [ ]:
volatility_summary = cv_summary.query("target_type == 'volatility'").sort_values(["symbol", "horizon", "mae"])
volatility_summary

,symbol,horizon,target_type,model,mae,mse,rmse,forecast_bias,n_observations,directional_accuracy,precision_positive,recall_positive,f1_positive
10,NG.L,5,volatility,volatility_historical_median,0.011341,0.000313,0.017686,0.000134,3472,NaN,NaN,NaN,NaN
11,NG.L,5,volatility,volatility_rolling_historical,0.011715,0.000310,0.017594,0.002810,3472,NaN,NaN,NaN,NaN
8,NG.L,5,volatility,volatility_GARCH11,0.013249,0.000293,0.017113,0.007933,3472,NaN,NaN,NaN,NaN
9,NG.L,5,volatility,volatility_historical_mean,0.015174,0.000381,0.019528,0.008361,3472,NaN,NaN,NaN,NaN
23,NG.L,10,volatility,volatility_rolling_historical,0.014175,0.000506,0.022492,0.002518,3472,NaN,NaN,NaN,NaN
22,NG.L,10,volatility,volatility_historical_median,0.014711,0.000518,0.022764,0.001433,3472,NaN,NaN,NaN,NaN
20,NG.L,10,volatility,volatility_GARCH11,0.017521,0.000504,0.022449,0.010941,3472,NaN,NaN,NaN,NaN
21,NG.L,10,volatility,volatility_historical_mean,0.021014,0.000686,0.026201,0.013139,3472,NaN,NaN,NaN,NaN
35,NG.L,20,volatility,volatility_rolling_historical,0.017795,0.000852,0.029182,0.002018,3472,NaN,NaN,NaN,NaN
34,NG.L,20,volatility,volatility_historical_median,0.019709,0.000876,0.029599,0.003943,3472,NaN,NaN,NaN,NaN


## 8. Interpretation, limitations and decision record

Interpret results relative to the naive forecasts and across folds, not by one metric alone. The development winner is subject to multiple-comparison and regime-selection bias; the one-time holdout is the relevant independent check for that chosen family. A holdout improvement alone does not establish economic value or profitability.

### Method limitations

- The row-count horizons proxy subsequent trading sessions; missing source rows are not calendar-repaired.
- Forward labels overlap. Purging prevents shared future-return intervals across a train/validation boundary, but does not make adjacent validation errors independent. Fold summaries are descriptive and uncertainty intervals are not yet estimated.
- The 20% holdout and feature families are design choices for this research iteration, not formally approved project thresholds.
- Features use closes at the forecast origin and assume the forecast is formed after that close. No same-session execution is modelled.
- Exact-date peer returns may share market shocks with the target. Incremental association does not establish independent causal information.
- Close-based returns exclude dividends; price-unit history, corporate actions and Yahoo Finance usage rights remain open risks.
- Macro features are omitted because historical point-in-time eligibility and licensing remain unresolved. FRED/ALFRED was not used.
- OLS coefficient interpretation is limited by feature redundancy and possible rank deficiency. No full-sample selection or tuning is performed.
- ARIMA is not fitted because the candidate target is already a forward return and a simpler AR(1) return benchmark suffices for this baseline pass. GARCH is not fitted here; conditional-variance residual diagnostics and a reproducible approved dependency path would be needed to justify it.
- No probability forecast is produced, so calibration, Brier score and log loss are not applicable in this notebook.

### Go/no-go recommendation

Carry forward a method only if it beats appropriate naive references on the untouched holdout, shows reasonably consistent development-fold behaviour, has acceptable errors and is reproducible. No numerical minimum is imposed. If no feature family reliably improves on zero/mean/median forecasts, retain the naive forecast as the benchmark and report a negative result. The project owner should approve any Phase 8 candidate and formal go/no-go threshold before ML/DL work begins.

### Reproducibility

This notebook uses fixed data acquisitions, deterministic OLS/least-squares calculations and no random seed. Source paths, target definitions, fold generation and feature groups are recorded above. Execution outputs should be saved with the notebook and summarised in the Phase 7 report.